#### YouTube Video RAG Chatbot Using Langchain

#### Necessary Imports

In [1]:
import os
from dotenv import load_dotenv
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEndpointEmbeddings,ChatHuggingFace, HuggingFaceEndpoint
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
# Load .env file having token to access the Huggingface API
load_dotenv()

# Get Hugging Face token
token = os.getenv("HUGGINGFACEHUB_API_TOKEN")
if not token:
    raise ValueError(
        "Hugging Face API token not found. Check your .env file."
    )

#print("Token loaded:", token is not None)

C:\Users\Dr. Shakeel\AppData\Local\Temp\ipykernel_14008\3359292132.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


#### Step 01 Indexing

In [2]:
video_id = "Gfr50f6ZBvo" # do not use full url .use just video id
try:
    # If you don’t care which language, this returns the “best” one
    api = YouTubeTranscriptApi()
    transcript_list = api.fetch(video_id, languages=["en"])

    # Flatten it to plain text
    transcript = " ".join(chunk.text for chunk in transcript_list)
    #print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

In [3]:
# text splitter
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [4]:
# Embeddings and vector store using Faiss
# Huggingface Embedding model
embeddings = HuggingFaceEndpointEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = FAISS.from_documents(
    chunks,
    embeddings
)
print('Embeddings stored in vector store FAISS')

Embeddings stored in vector store FAISS


In [5]:
#check any v
vector_store.get_by_ids(['61e45866-a18f-4e57-bfd1-1d3e20aba115'])

[]

#### Step 02 Retrievel formation

In [6]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})
retriever.invoke("what is deepmind")# example just to check retriver working or not

[Document(id='8efd2bc5-3653-47ce-a99c-6f4b5b95e729', metadata={}, page_content="and how it works this is tough to uh ask you this question because you probably will say it's everything but let's let's try let's try to think to this because you're in a very interesting position where deepmind is the place of some of the most uh brilliant ideas in the history of ai but it's also a place of brilliant engineering so how much of solving intelligence this big goal for deepmind how much of it is science how much is engineering so how much is the algorithms how much is the data how much is the hardware compute infrastructure how much is it the software computer infrastructure yeah um what else is there how much is the human infrastructure and like just the humans interact in certain kinds of ways in all the space of all those ideas how much does maybe like philosophy how much what's the key if um uh if if you were to sort of look back like if we go forward 200 years look back what was the key 

#### Step 03 Make a prompt template

In [7]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question'])

#### Step 04 Get the question and associated retrieved documents

In [8]:
question          = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs) # cancatenate the page content of from all the relevent retrieved docs
context_text

"in this case in fusion we we collaborated with epfl in switzerland the swiss technical institute who are amazing they have a test reactor that they were willing to let us use which you know i double checked with the team we were going to use carefully and safely i was impressed they managed to persuade them to let us use it and um and it's a it's an amazing test reactor they have there and they try all sorts of pretty crazy experiments on it and um the the the what we tend to look at is if we go into a new domain like fusion what are all the bottleneck problems uh like thinking from first principles you know what are all the bottleneck problems that are still stopping fusion working today and then we look at we you know we get a fusion expert to tell us and then we look at those bottlenecks and we look at the ones which ones are amenable to our ai methods today yes right and and and then and would be interesting from a research perspective from our point of view from an ai point of\n\

#### Step 05 Updtae the input varaibles of dynamic prompt to get final prompt

In [9]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

#### Step 06 LLM Model

In [10]:
# Hugging Face model
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    provider="featherless-ai",
    huggingfacehub_api_token=token,
    max_new_tokens=512,
    temperature=0.7
)

# Convert to LangChain chat model
model = ChatHuggingFace(llm=llm)
result=model.invoke(final_prompt)
print(result.content)

Yes, the topic of nuclear fusion is discussed in this video. The speaker mentions collaborating with EPFL in Switzerland to use their test reactor to address bottleneck problems in fusion. They also discuss successfully containing plasma in specific shapes for a record amount of time and how this relates to energy production. Additionally, they talk about using deep reinforcement learning for the control of high-temperature plasmas in nuclear fusion, as evidenced by a paper they have on this topic.


#### Doing the same using Chain

In [11]:
# necssary imports: runable for chain and parsers for final output
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

#### runablelamda needs function to covert info form our retrived docs into single context_text

In [12]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [13]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [14]:
parser = StrOutputParser()
main_chain = parallel_chain | prompt | model | parser
main_chain.invoke('Can you summarize the video')

"The video discusses the concept of a deeper, simpler explanation for physics that could encompass mysteries like consciousness, life, and gravity, potentially beyond the standard model of physics. It emphasizes the importance of clear and simple explanations, drawing on examples like Richard Feynman and Claude Shannon's early chess programs. The conversation highlights the symbiotic relationship between humans and technology, suggesting that enhancing our understanding and abilities through technology is an ongoing process."